# Predictions

Here we finally deploy our trained models to make predictions on real images.
The program uses Sliding Window method to extract sub-images from the full satellite image.
It then extracts features of those sub-images to determine the presence of waste in them.

In [2]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import time
import Utilities

In [11]:
src_folder_path = '/Users/sinner/Desktop/Batch3'
dest_folder_path = '/Users/sinner/Desktop/Batch_processing_outputs'

In [12]:
# Defining a custom standard scaler to scale features

mean_std_df = pd.read_csv('mean_std_df.csv')

def my_standard_scaler(features):
    scaled_features = []
    for i, value in enumerate(features):
        mean = mean_std_df.iat[i, 1]
        std = mean_std_df.iat[i, 2]
        scaled_features.append((value - mean) / std)
    return np.array(scaled_features)

In [13]:
# Load model
loaded_model = pickle.load(open('trained_models/RandomForestClassifier.sav', 'rb'))
print('Model Loaded')

Model Loaded


In [14]:
# Driver code

start_time = time.time()

# iterating in source folder
for filename in os.listdir(src_folder_path):

    # Check whether filetype is correct
    if not filename.endswith(('.jpg', '.png', '.jpeg')):
        continue

    print(f"Scanning {filename}")

    # load image
    img_bgr = cv2.imread(os.path.join(src_folder_path, filename))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    positiveWindows = []
    for window_coord in Utilities.get_window_coords(img_rgb, 150, 150):

        # get cropped window from img
        # cropped_img = img[top:bottom, left:right]
        window = img_rgb[round(window_coord[0]):round(window_coord[1]), round(window_coord[2]):round(window_coord[3])]

        # get features of window
        features = Utilities.get_features(window)
        # Feature Scaling
        features = my_standard_scaler(features)

        # Predict class using model
        predicted_class = loaded_model.predict(np.reshape(features, (1,-1)))
        # print(predicted_class)

        if predicted_class[0] == 1:
            # append coordinates into positiveWindows list if waste detected
            positiveWindows.append(window_coord)

        overlay_time_start = time.time()
        for window_coord in positiveWindows:
            # Overlay the bounding box on the image
            cv2.rectangle(img_bgr, (window_coord[2], window_coord[0]), (window_coord[3], window_coord[1]), (0, 255, 0), 2)

    cv2.imwrite(os.path.join(dest_folder_path, filename), img_bgr)

print(f"Total time taken: {time.time() - start_time}")

Scanning 0105.jpg
Scanning 0111.jpg
Scanning 0139.jpg
Scanning 0138.jpg
Scanning 0110.jpg
Scanning 0104.jpg
Scanning 0112.jpg
Scanning 0106.jpg
Scanning 0107.jpg
Scanning 0113.jpg
Scanning 0117.jpg
Scanning 0103.jpg
Scanning 0102.jpg
Scanning 0116.jpg
Scanning 0128.jpg
Scanning 0114.jpg
Scanning 0115.jpg
Scanning 0101.jpg
Scanning 0129.jpg
Scanning 0166.jpg
Scanning 0172.jpg
Scanning 0199.jpg
Scanning 0198.jpg
Scanning 0173.jpg
Scanning 0167.jpg
Scanning 0171.jpg
Scanning 0165.jpg
Scanning 0159.jpg
Scanning 0158.jpg
Scanning 0164.jpg
Scanning 0170.jpg
Scanning 0148.jpg
Scanning 0174.jpg
Scanning 0160.jpg
Scanning 0161.jpg
Scanning 0175.jpg
Scanning 0149.jpg
Scanning 0163.jpg
Scanning 0177.jpg
Scanning 0188.jpg
Scanning 0189.jpg
Scanning 0176.jpg
Scanning 0162.jpg
Scanning 0200.jpg
Scanning 0147.jpg
Scanning 0153.jpg
Scanning 0184.jpg
Scanning 0190.jpg
Scanning 0191.jpg
Scanning 0185.jpg
Scanning 0152.jpg
Scanning 0146.jpg
Scanning 0150.jpg
Scanning 0144.jpg
Scanning 0178.jpg
Scanning 0